# Index Strategies

## Overview
Indexes are the primary tool for query performance. Choosing the right index type, column order, and conditions can reduce query time from seconds to milliseconds on large tables.

**Index type summary (PostgreSQL):**

| Type | Best for | Notes |
|---|---|---|
| `B-tree` (default) | Equality, range, ORDER BY, LIKE 'prefix%' | Works for almost every column |
| `Hash` | Equality only (`=`) | Slightly faster than B-tree for equality; no range support |
| `GIN` | Arrays, JSONB, full-text search, multi-value columns | Slower to update; fast for containment queries |
| `GiST` | Geometric types, range types, PostGIS geometries | Flexible; supports custom operator classes |
| `BRIN` | Very large append-only tables where values correlate with storage order | Tiny index, coarse; good for timestamps on write-ordered tables |
| `SP-GiST` | Non-balanced tree structures (quad-trees, k-d trees) | Niche: spatial, IP range, text prefix lookups |

**Key index design rules:**
1. Index FK columns -- PostgreSQL does not auto-index them
2. Composite index column order: most selective column first, then filtering, then sorting
3. Use partial indexes to index only the rows you actually query
4. Use covering indexes (INCLUDE) to enable index-only scans

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE sites (site_id INTEGER PRIMARY KEY, site_name TEXT NOT NULL, region TEXT, latitude REAL, longitude REAL, elevation_m REAL, established_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE species (species_id INTEGER PRIMARY KEY, common_name TEXT NOT NULL, scientific_name TEXT NOT NULL UNIQUE, taxon_group TEXT, native INTEGER DEFAULT 1, at_risk INTEGER DEFAULT 0);CREATE TABLE field_crew (crew_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, role TEXT, certified INTEGER DEFAULT 1);CREATE TABLE observations (obs_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), species_id INTEGER REFERENCES species(species_id), crew_id INTEGER REFERENCES field_crew(crew_id), obs_date TEXT NOT NULL, count_individuals INTEGER, life_stage TEXT, method TEXT, notes TEXT);CREATE TABLE water_quality (wq_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), crew_id INTEGER REFERENCES field_crew(crew_id), sample_date TEXT NOT NULL, ph REAL, dissolved_o2 REAL, turbidity_ntu REAL, temp_c REAL, conductivity_us REAL);INSERT INTO sites VALUES (1,'Fundy Estuary','Atlantic',45.78,-64.52,3.2,'2018-04-01',1),(2,'Kejimkujik Lake','Atlantic',44.43,-65.21,156.0,'2017-06-15',1),(3,'Presqu ile Point','Great Lakes',43.99,-77.72,75.5,'2019-03-20',1),(4,'Rondeau Bay','Great Lakes',42.31,-81.87,176.0,'2018-09-10',1),(5,'Athabasca Delta','Boreal',58.72,-110.87,213.0,'2016-01-15',1),(6,'Wapusk Tundra','Boreal',57.92,-93.15,45.0,'2017-07-01',1),(7,'Clayoquot Sound','Pacific',49.15,-125.93,12.0,'2015-05-20',1),(8,'Boundary Bay','Pacific',49.01,-122.97,1.5,'2016-08-01',1),(9,'Lac Saint-Pierre','St Lawrence',46.19,-72.87,8.0,'2018-02-14',1),(10,'Baie des Chaleurs','Atlantic',48.01,-65.72,5.0,'2019-11-01',0);INSERT INTO species VALUES (1,'Atlantic Salmon','Salmo salar','Fish',1,1),(2,'Great Blue Heron','Ardea herodias','Bird',1,0),(3,'Wood Duck','Aix sponsa','Bird',1,0),(4,'Spotted Turtle','Clemmys guttata','Reptile',1,1),(5,'Common Loon','Gavia immer','Bird',1,0),(6,'Muskrat','Ondatra zibethicus','Mammal',1,0),(7,'Northern Pike','Esox lucius','Fish',1,0),(8,'Bald Eagle','Haliaeetus leucocephalus','Bird',1,0),(9,'American Bittern','Botaurus lentiginosus','Bird',1,1),(10,'Snapping Turtle','Chelydra serpentina','Reptile',1,0),(11,'Rainbow Trout','Oncorhynchus mykiss','Fish',0,0),(12,'Ring-necked Duck','Aythya collaris','Bird',1,0),(13,'Beaver','Castor canadensis','Mammal',1,0),(14,'River Otter','Lontra canadensis','Mammal',1,0),(15,'Painted Turtle','Chrysemys picta','Reptile',1,0);INSERT INTO field_crew VALUES (1,'Dr. Aroha Ngata','Biologist',1),(2,'Liam Chen','Technician',1),(3,'Fatima Al-Rashid','Biologist',1),(4,'James MacLeod','Technician',1),(5,'Sofia Petrov','Student',0);INSERT INTO observations VALUES (1,1,1,1,'2023-04-10',12,'Adult','Electrofishing',NULL),(2,1,2,2,'2023-04-10',3,'Adult','Visual',NULL),(3,2,5,1,'2023-04-15',2,'Adult','Auditory',NULL),(4,2,4,3,'2023-04-15',8,'Juvenile','Visual',NULL),(5,3,2,2,'2023-05-01',5,'Adult','Visual',NULL),(6,3,3,4,'2023-05-01',7,'Adult','Visual',NULL),(7,4,10,3,'2023-05-10',2,'Adult','Visual',NULL),(8,4,7,1,'2023-05-10',4,'Adult','Electrofishing',NULL),(9,5,13,2,'2023-05-20',6,'Adult','Camera Trap',NULL),(10,5,14,3,'2023-05-20',1,'Adult','Visual',NULL),(11,6,8,1,'2023-06-01',3,'Adult','Visual',NULL),(12,6,6,4,'2023-06-01',9,'Adult','Trap',NULL),(13,7,14,3,'2023-06-15',2,'Adult','Visual',NULL),(14,7,5,2,'2023-06-15',4,'Adult','Auditory',NULL),(15,8,2,1,'2023-07-01',12,'Adult','Visual',NULL),(16,8,8,4,'2023-07-01',2,'Adult','Visual',NULL),(17,9,1,3,'2023-07-10',7,'Adult','Electrofishing',NULL),(18,9,9,1,'2023-07-10',1,'Adult','Visual','First sighting this season'),(19,1,4,2,'2023-08-05',1,'Adult','Visual',NULL),(20,2,13,4,'2023-08-05',4,'Adult','Camera Trap',NULL),(21,3,6,3,'2023-08-12',11,'Adult','Trap',NULL),(22,4,2,2,'2023-08-12',7,'Adult','Visual',NULL),(23,5,8,1,'2023-09-01',1,'Adult','Visual',NULL),(24,6,14,3,'2023-09-01',3,'Adult','Visual',NULL),(25,7,1,4,'2023-09-15',18,'Adult','Electrofishing',NULL),(26,8,5,2,'2023-09-15',6,'Adult','Visual',NULL),(27,9,4,3,'2023-09-20',2,'Juvenile','Visual',NULL),(28,1,7,1,'2023-10-01',3,'Adult','Electrofishing',NULL),(29,2,1,2,'2023-10-01',9,'Adult','Electrofishing',NULL),(30,3,8,4,'2023-10-10',4,'Adult','Visual',NULL),(31,4,5,1,'2023-10-15',3,'Adult','Auditory',NULL),(32,5,2,3,'2023-10-15',8,'Adult','Visual',NULL);INSERT INTO water_quality VALUES (1,1,1,'2023-04-10',7.2,9.1,3.4,8.5,142.0),(2,1,2,'2023-06-15',7.4,8.6,4.2,14.2,138.0),(3,1,3,'2023-08-20',7.1,7.8,5.1,19.6,145.0),(4,2,1,'2023-04-15',6.8,10.2,1.2,7.1,98.0),(5,2,4,'2023-07-01',6.9,9.4,1.8,16.3,102.0),(6,3,2,'2023-05-01',7.6,9.8,2.3,11.4,188.0),(7,3,3,'2023-08-01',7.5,8.2,3.7,20.1,192.0),(8,4,1,'2023-05-10',7.8,9.5,1.9,13.0,205.0),(9,4,4,'2023-09-05',7.7,8.9,2.5,18.4,198.0),(10,5,2,'2023-05-20',7.3,10.8,0.8,9.2,55.0),(11,5,1,'2023-09-10',7.2,9.6,1.1,15.7,58.0),(12,6,3,'2023-06-01',6.5,11.2,0.5,6.8,32.0),(13,7,2,'2023-06-15',7.9,9.0,1.4,12.5,310.0),(14,8,4,'2023-07-01',7.8,8.7,2.1,17.2,295.0),(15,9,1,'2023-07-10',7.6,9.3,2.8,14.8,178.0);""")
conn.commit()
print("Ecological schema ready.")
for t in ["sites","species","field_crew","observations","water_quality"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

Ecological schema ready.
  sites: 10 rows
  species: 15 rows
  field_crew: 5 rows
  observations: 32 rows
  water_quality: 15 rows


---
## Index usage: EXPLAIN QUERY PLAN before and after

In [2]:
print("=== Index impact on observations lookups ===")

queries = [
    ("site_id lookup (FK -- should be indexed)",
     "SELECT * FROM observations WHERE site_id = 1"),
    ("obs_date range",
     "SELECT * FROM observations WHERE obs_date BETWEEN '2023-06-01' AND '2023-08-31'"),
    ("method equality",
     "SELECT * FROM observations WHERE method = 'Electrofishing'"),
]

for label, q in queries:
    plan_before = conn.execute(f"EXPLAIN QUERY PLAN {q}").fetchall()
    scan_before = " | ".join(str(r) for r in plan_before)
    print(f"  {label}")
    print(f"  Before index: {scan_before}")

# Create indexes
conn.executescript("""
    CREATE INDEX idx_obs_site    ON observations (site_id);
    CREATE INDEX idx_obs_date    ON observations (obs_date);
    CREATE INDEX idx_obs_method  ON observations (method);
    CREATE INDEX idx_obs_species ON observations (species_id);
    CREATE INDEX idx_wq_site     ON water_quality (site_id);
    CREATE INDEX idx_wq_date     ON water_quality (sample_date);
""")
conn.commit()

print()
for label, q in queries:
    plan_after = conn.execute(f"EXPLAIN QUERY PLAN {q}").fetchall()
    scan_after = " | ".join(str(r) for r in plan_after)
    print(f"  {label}")
    print(f"  After index:  {scan_after}")
    print()


=== Index impact on observations lookups ===
  site_id lookup (FK -- should be indexed)
  Before index: (2, 0, 0, 'SCAN observations')
  obs_date range
  Before index: (2, 0, 0, 'SCAN observations')
  method equality
  Before index: (2, 0, 0, 'SCAN observations')

  site_id lookup (FK -- should be indexed)
  After index:  (3, 0, 0, 'SEARCH observations USING INDEX idx_obs_site (site_id=?)')

  obs_date range
  After index:  (3, 0, 0, 'SEARCH observations USING INDEX idx_obs_date (obs_date>? AND obs_date<?)')

  method equality
  After index:  (3, 0, 0, 'SEARCH observations USING INDEX idx_obs_method (method=?)')



---
## Composite indexes and the leftmost prefix rule

In [3]:
print("=== Composite index: (site_id, obs_date, method) ===")
conn.execute("CREATE INDEX idx_obs_composite ON observations (site_id, obs_date, method)")
conn.commit()

test_cases = [
    ("WHERE site_id = 1",                                    "uses index (leftmost prefix)"),
    ("WHERE site_id = 1 AND obs_date > '2023-06-01'",        "uses index (site_id + obs_date)"),
    ("WHERE site_id = 1 AND obs_date > '2023-06-01' AND method = 'Visual'",
                                                              "uses full composite index"),
    ("WHERE obs_date > '2023-06-01'",                        "skips index (no leftmost prefix site_id)"),
    ("WHERE method = 'Visual'",                              "skips index (no leftmost prefix)"),
]

for clause, expectation in test_cases:
    q = f"EXPLAIN QUERY PLAN SELECT obs_id FROM observations {clause}"
    plan = conn.execute(q).fetchall()
    uses = any("idx_obs_composite" in str(r) for r in plan)
    status = "USES composite" if uses else "skips composite"
    print(f"  {clause:<55s} -> {status}  ({expectation})")


=== Composite index: (site_id, obs_date, method) ===
  WHERE site_id = 1                                       -> skips composite  (uses index (leftmost prefix))
  WHERE site_id = 1 AND obs_date > '2023-06-01'           -> USES composite  (uses index (site_id + obs_date))
  WHERE site_id = 1 AND obs_date > '2023-06-01' AND method = 'Visual' -> USES composite  (uses full composite index)
  WHERE obs_date > '2023-06-01'                           -> skips composite  (skips index (no leftmost prefix site_id))
  WHERE method = 'Visual'                                 -> skips composite  (skips index (no leftmost prefix))


---
## Partial indexes and covering indexes

In [4]:
print("=== Partial index: only at-risk species observations ===")
# A partial index only covers rows matching the WHERE clause
# Smaller, faster to maintain, faster for those specific queries
conn.execute("""
    CREATE INDEX idx_obs_at_risk ON observations (site_id, obs_date)
    WHERE species_id IN (1, 4, 9)   -- Atlantic Salmon, Spotted Turtle, American Bittern
""")
conn.commit()

plan = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT obs_id, obs_date, count_individuals
    FROM   observations
    WHERE  site_id = 1
      AND  species_id IN (1, 4, 9)
""").fetchall()
print("Partial index query plan:")
for row in plan:
    print(" ", row)

print()
print("=== Covering index (INCLUDE in PostgreSQL) ===")
print("""
PostgreSQL only -- INCLUDE lets you add extra columns to the index leaf nodes
so the query can be answered entirely from the index (Index Only Scan):

  CREATE INDEX idx_obs_covering
      ON observations (site_id, obs_date)
      INCLUDE (count_individuals, method);

  -- Query answered from index alone (no heap access):
  SELECT obs_date, count_individuals, method
  FROM   observations
  WHERE  site_id = 1
  ORDER BY obs_date;

  -- EXPLAIN shows: Index Only Scan using idx_obs_covering
  -- Heap Fetches: 0  <-- no table access needed
""")

print("=== All indexes on observations ===")
idx = conn.execute(
    "SELECT name, sql FROM sqlite_master WHERE type='index' AND tbl_name='observations'"
).fetchall()
for name, sql in idx:
    if sql:
        print(f"  {name}: {sql[:80]}")

conn.close()


=== Partial index: only at-risk species observations ===
Partial index query plan:
  (3, 0, 0, 'SEARCH observations USING INDEX idx_obs_at_risk (site_id=?)')

=== Covering index (INCLUDE in PostgreSQL) ===

PostgreSQL only -- INCLUDE lets you add extra columns to the index leaf nodes
so the query can be answered entirely from the index (Index Only Scan):

  CREATE INDEX idx_obs_covering
      ON observations (site_id, obs_date)
      INCLUDE (count_individuals, method);

  -- Query answered from index alone (no heap access):
  SELECT obs_date, count_individuals, method
  FROM   observations
  WHERE  site_id = 1
  ORDER BY obs_date;

  -- EXPLAIN shows: Index Only Scan using idx_obs_covering
  -- Heap Fetches: 0  <-- no table access needed

=== All indexes on observations ===
  idx_obs_site: CREATE INDEX idx_obs_site    ON observations (site_id)
  idx_obs_date: CREATE INDEX idx_obs_date    ON observations (obs_date)
  idx_obs_method: CREATE INDEX idx_obs_method  ON observations (method)

---
## Common Pitfalls

**1. Not indexing foreign key columns**
PostgreSQL creates an index on the PRIMARY KEY automatically, but NOT on foreign key columns. `observations.site_id` has no index unless you create one explicitly. Every join on `observations.site_id = sites.site_id` does a full sequential scan of observations. Always create indexes on FK columns.

**2. Creating many single-column indexes instead of one composite index**
If queries always filter on `(site_id, obs_date)` together, two separate indexes are less efficient than one composite index. The planner may combine them with a Bitmap And plan, but a purpose-built composite index is faster and uses less space.

**3. Composite index column order ignoring the leftmost prefix rule**
`CREATE INDEX ON observations (method, site_id)` cannot be used for `WHERE site_id = 1` alone -- method is the leftmost column so only queries filtering on method first benefit. Put the column most commonly used in equality filters first.

**4. Partial index WHERE clause not matching the query exactly**
`CREATE INDEX idx_at_risk ON observations (site_id) WHERE at_risk = 1` cannot be used by `WHERE at_risk = true` or `WHERE at_risk != 0` -- the condition must match exactly. In PostgreSQL, the partial index WHERE must be an immutable expression that the planner can verify matches the query predicate.

**5. Covering index columns put in the main key instead of INCLUDE**
`CREATE INDEX ON observations (site_id, obs_date, count_individuals, method)` puts count_individuals and method in the B-tree key, affecting index size and update cost. Use `INCLUDE (count_individuals, method)` to store them in leaf nodes only -- smaller key, same index-only scan benefit.

**6. Indexes on very low-cardinality columns without partial indexing**
A B-tree index on `life_stage` (Adult/Juvenile) with 2 values is almost never used by the planner -- a seq scan of half the table is faster than following index pointers to half the rows. Either drop this index or use a partial index with a specific value: `WHERE life_stage = 'Juvenile'`.


---
*sql_methods_library - Samantha McGarrigle*